# 13 · `gl_engine/schema/fields.py`

## What this file is for

**The submission format, read from ISO rather than designed by us.**

Somebody had to decide what a submission looks like — which fields exist, which are required, what values are legal. The tempting move is to design that. This file doesn't: it reads ISO's own `Fields.FormField.csv` for the resolved jurisdiction, so the schema *is* whatever ISO declared, per state, per edition.

That is why there is no per-state input format anywhere in the code, and why a field that only Georgia requires simply appears when you resolve Georgia.

**Depends on:** [`06-resolve-book`](06-resolve-book.ipynb), [`07-erc-tables`](07-erc-tables.ipynb).

## Its public surface

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import inspect
from gl_engine.schema import fields

for name, obj in vars(fields).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != fields.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                print(f"    .{m}{inspect.signature(f)}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

Load the schema ISO declares for one state on one date.

In [ ]:
from gl_engine import EditionResolver
from gl_engine.resolve.book import ResolvedBook

book = ResolvedBook(EditionResolver().resolve("GA", "20260811"))

from gl_engine.schema.fields import Schema

schema = Schema.for_book(book)
print("fields declared :", len(schema.tables()) and sum(1 for _ in schema.tables()))
print("tables          :", len(schema.tables()))
print("required (uncond):", len(schema.required()))
print()
for t in schema.tables()[:6]:
    print("  ", t)

## The interesting case

### A field carries ISO's own metadata, not ours

In [ ]:
f = schema.required()[0]
for k, v in vars(f).items():
    if v not in ("", None):
        print(f"{k:<20} {v!r}")

`policy_required` and `quote_required` differ, and `condition` holds ISO's own rule for *when* a field becomes required. That condition is deliberately **not** evaluated — the dialect isn't implemented, and guessing would demand a field ISO doesn't want. So only unconditionally-required fields are reported as required.

### Legal values come from ISO's domain tables

In [ ]:
import itertools

named = [f for f in schema.required(unconditional_only=False) if f.domain][:8]
for f in named:
    try:
        vals = schema.legal_values(f.table, f.column)
    except Exception as e:
        print(f"{f.column:<34} ({type(e).__name__})")
        continue
    show = ", ".join(str(v) for v in itertools.islice(vals, 3))
    print(f"{f.column:<34} {len(vals):>4} legal values   {show}...")

This is what the dropdowns in any tester are built from, and it is why a value can be refused *before* anything is sent anywhere: if ISO doesn't declare it for this jurisdiction, it isn't offered.

### The schema is per jurisdiction

Same date, two states, different declared surfaces.

In [ ]:
for juris in ("GA", "NY"):
    b = ResolvedBook(EditionResolver().resolve(juris, "20260811"))
    s = Schema.for_book(b)
    print(f"{juris}: {len(s.tables()):>3} tables, {len(s.required()):>3} unconditionally required")

## What it refuses

Asking for the legal values of something ISO doesn't declare.

In [ ]:
try:
    schema.legal_values("NoSuchTable", "NoSuchColumn")
    print("no error")
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:110]}")

## Try it yourself

1. How many of the declared fields are deductible fields? Compare `GA` with `NY`.
2. Find a field whose `condition` is non-empty. What is ISO's dialect saying, and why isn't it evaluated?
3. `CLASS_BASIS_CLIFF` from notebook 01 matters here. Resolve either side of it and see what moves.

In [ ]:
# your turn